# specdec-bench on Google Colab

EAGLE-3 / adaptive tree speculative decoding on Llama-3.1-8B-Instruct.

**Before you start (once):**
1. *Runtime → Change runtime type →* **A100 GPU** (needs Colab Pro, Pro+ or compute units). 40 GB is enough.
   An L4 (24 GB) works for Path B (EAGLE repo / adaptive tree) only. A T4 is too small (fp16 weights alone are 16 GB).
2. Accept the Llama 3.1 licence at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, create a read token,
   and add it in Colab's **🔑 Secrets** panel as `HF_TOKEN` (toggle *Notebook access* on).
3. Upload `specdec-bench.zip` to Google Drive at `MyDrive/specdec/specdec-bench.zip`.

**Every session:** run cells 1–4. The Colab machine is wiped when a session ends, so everything that must survive is
kept on the persistent disk (`STORE`, default `MyDrive/specdec`):

| What | Where on the disk | How |
| --- | --- | --- |
| Results + run checkpoints | `results/` | written there directly; flushed after **every question** |
| Model checkpoints (Llama-3.1-8B + both EAGLE-3 drafts, ~17 GB) | `models/*.tar` | saved automatically by cell 4 the first time, restored in later sessions |
| Your code (incl. any edits) | `code/specdec-bench.zip` | cell 12 |
| Installed environments (optional, ~10 GB each) | `envs/*.tar` | cell 12 with `SAVE_ENVS=1`; restored by cell 4 |

The models need ~17 GB, more than the free 15 GB Drive: use a Google One plan (100 GB) or set `SAVE_MODELS=0` in cell 2
(models are then re-downloaded each session, ~2–5 min).

**After a disconnect:** re-run cells 1–4 and then the experiment cell you were on. Finished result files are skipped, and a
half-finished one continues from the next unfinished question (it restarts only if the settings or GPU type changed,
so timings are never mixed).

In [4]:
# 1) GPU check
!nvidia-smi
import torch, subprocess, re
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> A100 GPU"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
drv = re.search(r"CUDA Version: (\d+)\.(\d+)", subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
print(f"GPU: {name}, {mem:.0f} GiB | driver CUDA {drv.group(1)}.{drv.group(2)}")
if mem < 22: print("WARNING: under 22 GiB. Llama-3.1-8B fp16 will not fit. Switch to an A100 or L4 runtime.")
elif mem < 38: print("NOTE: OK for Path B (EAGLE repo / adaptive tree). Path A (SGLang, batch 8/32) wants an A100.")

Tue Oct 20 16:43:12 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             42W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
# 2) Drive, Hugging Face token, paths
import os
from google.colab import drive, userdata
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/specdec"
os.makedirs(f"{DRIVE}/results", exist_ok=True)
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # the secret's NAME; read by huggingface_hub in every subprocess
except userdata.SecretNotFoundError:
    raise SystemExit("Add a secret named HF_TOKEN in the key-icon panel (left sidebar), switch on Notebook access, re-run.")
except userdata.NotebookAccessError:
    raise SystemExit("Switch on 'Notebook access' for HF_TOKEN in the key-icon panel (left sidebar), then re-run.")
os.environ["HF_HOME"] = "/content/hf_cache"             # local disk: fast, re-downloaded each session (~16 GB)
os.environ["DRIVE"] = DRIVE
STORE = DRIVE                      # persistent disk for checkpoints, code and envs (any mounted folder works)
SAVE_MODELS = 1                    # 0 = do not keep the ~17 GB of model weights on the disk
SAVE_ENVS = 0                      # 1 = cell 12 also saves the installed environments (~10 GB each)
os.makedirs(STORE, exist_ok=True)
os.environ.update(STORE=STORE, SAVE_MODELS=str(SAVE_MODELS), SAVE_ENVS=str(SAVE_ENVS))
print("results go to", f"{DRIVE}/results", "| persistent store:", STORE)

Mounted at /content/drive
results go to /content/drive/MyDrive/specdec/results | persistent store: /content/drive/MyDrive/specdec


In [2]:
%%bash
# 3) Code: unzip the newest copy (the zip you uploaded, or the one cell 12 saved), keep results on Drive
set -e
: "${DRIVE:?DRIVE is not set - cell 2 did not finish. Fix cell 2 (HF_TOKEN secret) and run it again.}"
cd /content
UP="$DRIVE/specdec-bench.zip"
[ -f "$UP" ] || [ -f "$STORE/code/specdec-bench.zip" ] || { echo "No code zip: upload specdec-bench.zip to $UP"; ls -la "$DRIVE"; exit 1; }; SAVED="$STORE/code/specdec-bench.zip"
SRC="$UP"; [ -f "$SAVED" ] && { [ ! -f "$UP" ] || [ "$SAVED" -nt "$UP" ]; } && SRC="$SAVED"
if [ ! -d specdec-bench ]; then
  echo "code from $SRC"; unzip -q "$SRC" -d /content
fi
cd specdec-bench
rm -rf results && ln -s "$DRIVE/results" results
ls

code from /content/drive/MyDrive/specdec/specdec-bench.zip
configs
envs
notebooks
README.md
results
scripts
specdec
tests


In [3]:
%%bash
# 4) Restore saved environments + model checkpoints, else install / download (then save the models)
set -e
cd /content/specdec-bench
python3 scripts/persist.py restore --store "$STORE" --what envs,models
# Path B environment (EAGLE repo @ pinned commit, torch 2.6.0 cu124, transformers 4.53.3): ~5 min
touch /content/setup_eagle.log
if [ ! -x .venv-eagle/bin/python ]; then
  bash envs/setup_eagle_env.sh > /content/setup_eagle.log 2>&1 || { tail -40 /content/setup_eagle.log; exit 1; }
fi
tail -2 /content/setup_eagle.log
source .venv-eagle/bin/activate
EAGLE_REPO=third_party/EAGLE python -m unittest discover -s tests 2>&1 | tail -3   # incl. adaptive tree on real torch + EAGLE source pins
deactivate
# download anything not restored (so no run is timed with a download) and keep a copy on the disk
if [ "$SAVE_MODELS" = 1 ]; then
  python3 scripts/persist.py save --store "$STORE" --what models \
    || echo "WARNING: could not save the models to $STORE (disk full?). They still work for this session."
else
  python3 -c "from huggingface_hub import snapshot_download as s; [s(r, ignore_patterns=['original/*','*.pth']) for r in ['meta-llama/Llama-3.1-8B-Instruct','yuhuili/EAGLE3-LLaMA3.1-Instruct-8B','lmsys/sglang-EAGLE3-LLaMA3.1-Instruct-8B']]"
fi
python3 scripts/persist.py status --store "$STORE"

[envs] eagle: not saved yet
[envs] sglang: not saved yet
[models] meta-llama/Llama-3.1-8B-Instruct: not saved yet (it will download from Hugging Face)
[models] yuhuili/EAGLE3-LLaMA3.1-Instruct-8B: not saved yet (it will download from Hugging Face)
[models] lmsys/sglang-EAGLE3-LLaMA3.1-Instruct-8B: not saved yet (it will download from Hugging Face)
torch 2.6.0+cu124 cuda True | transformers 4.53.3 | EaModel import OK
export EAGLE_REPO=/content/specdec-bench/third_party/EAGLE   # data dir: $EAGLE_REPO/eagle/data
Ran 53 tests in 35.043s

OK (skipped=3)
[models] meta-llama/Llama-3.1-8B-Instruct: saved 0.0 GiB in 0s -> /content/drive/MyDrive/specdec/models/models--meta-llama--Llama-3.1-8B-Instruct.tar
[models] yuhuili/EAGLE3-LLaMA3.1-Instruct-8B: saved 0.0 GiB in 0s -> /content/drive/MyDrive/specdec/models/models--yuhuili--EAGLE3-LLaMA3.1-Instruct-8B.tar
[models] lmsys/sglang-EAGLE3-LLaMA3.1-Instruct-8B: saved 0.0 GiB in 0s -> /content/drive/MyDrive/specdec/models/models--lmsys--sglang-EAGL

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 17.93it/s]


## Path B: EAGLE-3 and adaptive tree at batch size 1

In [5]:
%%bash
# 5) Smoke test: one prompt, plain decoding vs EAGLE-3 (K=10, D=8, N=60) vs adaptive tree (N=60, mu=0.03, K=10)
cd /content/specdec-bench && source .venv-eagle/bin/activate
python scripts/quickstart_adaptive.py


=== adaptive tree output ===
**Fibonacci Function**

The Fibonacci sequence is a series of numbers where a number is the addition of the last two numbers, starting with 0 and 1.

### Code

```python
def fibonacci(n):
    """
    Returns the n-th Fibonacci number.

    Args:
        n (int): The position of the Fibonacci number to return.

    Returns:
        int: The n-th Fibonacci number.

    Raises:
        ValueError: If n is a negative integer.

    Examples:
        >>> fibonacci(0)
        0
        >>> fibonacci(1)
        1
        >>> fibonacci(2)
        1
        >>> fibonacci(10)
        55
    """
    if not isinstance(n, int):
        raise TypeError("n must be an integer")
    if n < 0:
        raise ValueError("n must be a non-negative integer")
    elif n <= 1:
        return n
    else:
        return fibonacci(n-1) + fibonacci(n-2)
```

### Tests

```python
import unittest

class TestFibonacciFunction(unittest.TestCase):
    def test_fibonacci_zero(self):
        

LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.14s/it]
LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly

In [6]:
%%bash
# 6) Small real run before the long ones: 10 MT-Bench questions, EAGLE-3 + AR, then the losslessness check
cd /content/specdec-bench && source .venv-eagle/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data
python scripts/run_eagle.py --data-dir $EAGLE_DATA --bench mt_bench --limit 10 --with-ar \
    --steps 9 --topk 10 --budget 60 --out-dir results/colab_smoke --skip-existing --eagle-repo third_party/EAGLE
python scripts/check_lossless.py results/colab_smoke/mt_bench__s9_k10_b60__T0.jsonl results/colab_smoke/mt_bench__ar__T0.jsonl

[mt_bench:spec] 20 turns, 80.5 tok/s, tau=6.198, delta=9.18 -> results/colab_smoke/mt_bench__s9_k10_b60__T0.jsonl
[mt_bench:ar] 20 turns, 19.5 tok/s -> results/colab_smoke/mt_bench__ar__T0.jsonl
compared 20 turns by token ids: 13 identical (65.0%), 7 differ
first turns only: 8/10 identical
  mt_bench:82:1: first difference at tokens 209 (lengths 382 vs 404)
  mt_bench:84:0: first difference at tokens 229 (lengths 404 vs 354)
  mt_bench:84:1: first difference at tokens 39 (lengths 444 vs 340)
  mt_bench:86:1: first difference at tokens 134 (lengths 222 vs 294)
  mt_bench:87:1: first difference at tokens 202 (lengths 203 vs 249)


LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly defined. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.
Loading checkpoint shards: 100%|██████████| 4/4 [00:05<00:00,  1.28s/it]
LlamaForCausalLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly

In [11]:
%%bash
cd /content/specdec-bench && source .venv-eagle/bin/activate
python scripts/diagnose_mismatch.py \
    results/colab_smoke/mt_bench__s9_k10_b60__T0.jsonl \
    results/colab_smoke/mt_bench__ar__T0.jsonl \
    --data-dir $PWD/third_party/EAGLE/eagle/data

   mt_bench:82:1 @tok  209: spec ',' (rank 1, p=0.499) vs plain ' to' (rank 2, p=0.499), logit gap 0.000 -> fp16 near-tie
   mt_bench:84:0 @tok  229: spec ' They' (rank 2, p=0.216) vs plain ' You' (rank 1, p=0.216), logit gap 0.000 -> fp16 near-tie
   mt_bench:84:1: inherited (earlier turn differed)
   mt_bench:86:1 @tok  134: spec ' w' (rank 2, p=0.411) vs plain ' local' (rank 1, p=0.411), logit gap 0.000 -> fp16 near-tie
   mt_bench:87:1 @tok  202: spec '.' (rank 2, p=0.494) vs plain '.\n' (rank 1, p=0.494), logit gap 0.000 -> fp16 near-tie
   mt_bench:88:0 @tok   66: spec ' it' (rank 1, p=0.311) vs plain ' that' (rank 2, p=0.306), logit gap 0.016 -> fp16 near-tie
   mt_bench:88:1: inherited (earlier turn differed)

7 differing turns: 5 near-ties, 2 inherited, 0 suspicious


Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.13s/it]


In [12]:
%%bash
# 6c) Same 10 MT-Bench questions with adaptive tree (N=60, mu=0.03, K=10), then compare EAGLE-3 vs adaptive tree
#     against the plain-decoding run from cell 6 (~5 min; cell 6 must have finished).
cd /content/specdec-bench && source .venv-eagle/bin/activate
D=results/colab_smoke
python scripts/run_eagle.py --data-dir $PWD/third_party/EAGLE/eagle/data --bench mt_bench --limit 10 \
    --adaptive --budget 60 --mu 0.03 --root-k 10 --out-dir $D --skip-existing --eagle-repo third_party/EAGLE 2>&1 | grep -v -e GenerationMixin -e "^  - "
echo; echo "== adaptive tree vs plain decoding (losslessness) =="
python scripts/check_lossless.py $D/mt_bench__adaptive_n60_mu0.03_k10__T0.jsonl $D/mt_bench__ar__T0.jsonl
echo; echo "== EAGLE-3 vs adaptive tree on the same 20 answers =="
python - <<'PY'
import sys; sys.path.insert(0, ".")
from specdec.eagle_runner import read_jsonl
from specdec.metrics import throughput
D = "results/colab_smoke/mt_bench__"
runs = {"plain": "ar", "EAGLE-3 (K=10,D=8,N=60)": "s9_k10_b60", "adaptive tree (N=60,mu=0.03)": "adaptive_n60_mu0.03_k10"}
rows = {k: read_jsonl(f"{D}{v}__T0.jsonl")[1] for k, v in runs.items()}
ar = throughput(rows["plain"])
print(f"{'method':26s} {'tok/s':>7s} {'speedup':>8s} {'tau':>6s} {'delta':>6s} {'depth':>6s}")
for k, rs in rows.items():
    tps = throughput(rs)
    if k == "plain":
        print(f"{k:26s} {tps:7.1f} {1:7.2f}x"); continue
    tau = sum(r["new_tokens"] for r in rs) / sum(r["verify_steps"] for r in rs)
    delta = sum(r["draft_forwards"] for r in rs) / sum(r["verify_steps"] for r in rs)
    depth = sum(r["mean_tree_depth"] for r in rs) / len(rs)
    print(f"{k:26s} {tps:7.1f} {tps / ar:7.2f}x {tau:6.2f} {delta:6.2f} {depth:6.1f}")
PY

Loading checkpoint shards: 100%|██████████| 4/4 [00:04<00:00,  1.12s/it]
[mt_bench:spec] 20 turns, 80.7 tok/s, tau=5.888, delta=6.58 -> results/colab_smoke/mt_bench__adaptive_n60_mu0.03_k10__T0.jsonl

== adaptive tree vs plain decoding (losslessness) ==
compared 20 turns by token ids: 9 identical (45.0%), 11 differ
first turns only: 6/10 identical
  mt_bench:81:0: first difference at tokens 682 (lengths 770 vs 808)
  mt_bench:81:1: first difference at tokens 78 (lengths 800 vs 664)
  mt_bench:83:0: first difference at tokens 208 (lengths 219 vs 216)
  mt_bench:83:1: first difference at tokens 4 (lengths 38 vs 41)
  mt_bench:85:0: first difference at tokens 163 (lengths 193 vs 193)

== EAGLE-3 vs adaptive tree on the same 20 answers ==
method                       tok/s  speedup    tau  delta  depth
plain                         19.5    1.00x
EAGLE-3 (K=10,D=8,N=60)       80.5    4.12x   6.20   9.18    8.4
adaptive tree (N=60,mu=0.03)          80.7    4.13x   5.89   6.58    6.2


In [ ]:
%%bash
# 7) adaptive tree suite: paper setting + budget (N) and threshold (mu) ablations. Many hours on an A100.
#    Resumable: re-run this cell after a disconnect; finished files are skipped and the
#    interrupted one continues from its last finished question.
#    Log is on Drive: MyDrive/specdec/results/adaptive_log.txt
cd /content/specdec-bench && source .venv-eagle/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data
bash scripts/run_adaptive_suite.sh results/adaptive 2>&1 | tee -a results/adaptive_log.txt

## Path A: SGLang at batch sizes 1 / 8 / 32 (A100 recommended)

Uses a separate environment. SGLang `v0.5.22` needs a CUDA 13 driver; `v0.5.19` is the last release for CUDA 12.
The cell picks the tag from the driver version printed in cell 1.

In [ ]:
%%bash
# 8) Path A environment (~5-10 min)
set -e
cd /content/specdec-bench
DRV=$(nvidia-smi | grep -oP 'CUDA Version: \K[0-9]+')
if [ "$DRV" -ge 13 ]; then export SGLANG_TAG=v0.5.22; else export SGLANG_TAG=v0.5.19; fi
echo "driver CUDA $DRV -> SGLang $SGLANG_TAG"
touch /content/setup_sglang.log
if [ ! -x .venv-sglang/bin/python ]; then
  bash envs/setup_sglang_env.sh > /content/setup_sglang.log 2>&1 || { tail -40 /content/setup_sglang.log; exit 1; }
fi
tail -1 /content/setup_sglang.log
# question files come from the EAGLE checkout (made by cell 4); clone it if you skipped Path B
[ -d third_party/EAGLE ] || git clone -q https://github.com/SafeAILab/EAGLE.git third_party/EAGLE

In [ ]:
%%bash
# 9) SGLang smoke: 2 servers (AR, EAGLE-3 at budget 60), 10 questions, batch 1 and 8
cd /content/specdec-bench && source .venv-sglang/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data
python scripts/run_sglang_sweep.py --config configs/smoke.yaml --data-dir $EAGLE_DATA \
    --out-dir results/sglang_smoke --skip-existing 2>&1 | tail -30

In [ ]:
%%bash
# 10) Full matched-budget sweep (21 servers x 6 benchmarks x batch 1/8/32). Many hours; resumable.
cd /content/specdec-bench && source .venv-sglang/bin/activate
export EAGLE_DATA=$PWD/third_party/EAGLE/eagle/data
python scripts/run_sglang_sweep.py --config configs/sweep_eagle3_llama31.yaml --data-dir $EAGLE_DATA \
    --out-dir results/sglang --skip-existing 2>&1 | tee -a results/sglang_log.txt

## Results

In [ ]:
%%bash
# 11) One CSV over everything finished so far (saved to Drive)
cd /content/specdec-bench && source .venv-eagle/bin/activate
python scripts/summarize.py results/ --out results/summary.csv

In [ ]:
import pandas as pd
df = pd.read_csv(f"{DRIVE}/results/summary.csv")
pd.set_option("display.max_rows", 200, "display.max_columns", 40)
df

## Save to disk

In [ ]:
%%bash
# 12) Save code (with any edits you made here) and, if SAVE_ENVS=1, the installed environments.
#     Results and model checkpoints are already on the disk.
cd /content/specdec-bench
WHAT=code; [ "$SAVE_ENVS" = 1 ] && WHAT=code,envs
python3 scripts/persist.py save --store "$STORE" --what $WHAT
python3 scripts/persist.py status --store "$STORE"